In [1]:
import base64
import csv
import hashlib
import io
import json
import re
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import zipfile
from datetime import datetime, timezone
from pathlib import Path

MAIN_NAME = "Lab1_MusicGen.ipynb"
candidates = [Path.cwd(), *Path.cwd().parents,
              Path.cwd() / "АнДан" / "Лабораторная 1"]
ROOT = next((path for path in candidates if (path / MAIN_NAME).is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f"Положите {MAIN_NAME} рядом с этим ноутбуком")
DATA = ROOT / ".lab1" / "data"
saved_notebook = json.loads((ROOT / MAIN_NAME).read_text(encoding="utf-8"))
payload = base64.b85decode(saved_notebook["metadata"]["embedded_sources_b85"])
with zipfile.ZipFile(io.BytesIO(payload)) as saved:
    for name in saved.namelist():
        if not name.startswith("data/") or name.endswith("/"):
            continue
        target = (ROOT / ".lab1" / name).resolve()
        if not target.is_relative_to((ROOT / ".lab1").resolve()):
            raise ValueError("Некорректный путь в архиве")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(saved.read(name))
expected_papers = [json.loads(line) for line in
                   (DATA / "papers.jsonl").read_text(encoding="utf-8").splitlines()]
expected_manifest = json.loads((DATA / "collection_manifest.json").read_text(encoding="utf-8"))
expected_search = json.loads((DATA / "discovery" / "api_search_manifest.json").read_text(encoding="utf-8"))
expected_review = (DATA / "selection_review.csv").read_text(encoding="utf-8-sig")
print(f"Исходный снимок: {len(expected_papers)} публикации, "
      f"{len(expected_manifest['api_responses'])} ответов API")

Исходный снимок: 793 публикации, 17 ответов API


In [2]:
RAW = DATA / 'discovery' / 'arxiv_api'
RAW.mkdir(parents=True, exist_ok=True)
NS = {'a': 'http://www.w3.org/2005/Atom', 'o': 'http://a9.com/-/spec/opensearch/1.1/'}
QUERY = ('(ti:music OR abs:music OR ti:musical OR abs:musical) AND '
         '(ti:generation OR ti:generative OR ti:generating OR ti:composition OR '
         'ti:composing OR ti:synthesis OR ti:composer) AND '
         'submittedDate:[202001010000 TO 202610112359]')
responses = []
used_network = False
records = {}
start = 0
while True:
    params = {'search_query': QUERY, 'start': start, 'max_results': 100,
              'sortBy': 'submittedDate', 'sortOrder': 'ascending'}
    url = 'https://export.arxiv.org/api/query?' + urllib.parse.urlencode(params)
    path = RAW / f'search_{start:04d}.xml'
    if path.exists():
        content = path.read_bytes()
    else:
        used_network = True
        for attempt in range(3):
            try:
                req = urllib.request.Request(url, headers={'User-Agent': 'ITMO-MusicGenerationLab/1.0',
                                                           'Accept': 'application/atom+xml'})
                with urllib.request.urlopen(req, timeout=45) as response:
                    content = response.read()
                ET.fromstring(content)
                path.write_bytes(content)
                break
            except Exception as error:
                print('Retry', start, attempt + 1, type(error).__name__, str(error), flush=True)
                if attempt == 2:
                    raise
                time.sleep(10 * (attempt + 1))
            finally:
                time.sleep(3.5)
    feed = ET.fromstring(content)
    total = int(feed.findtext('o:totalResults', '0', NS))
    entries = feed.findall('a:entry', NS)
    for entry in entries:
        match = re.search(r'(\d{4}\.\d{4,5})(v\d+)?$', entry.findtext('a:id', '', NS))
        if match is None:
            raise ValueError('Invalid article ID')
        aid = match.group(1)
        clean = lambda tag: ' '.join(entry.findtext('a:' + tag, '', NS).split())
        row = {'arxiv_id': aid, 'arxiv_version': match.group(2), 'title': clean('title'),
               'abstract': clean('summary'), 'authors': [a.findtext('a:name', '', NS) for a in entry.findall('a:author', NS)],
               'published': clean('published'), 'updated': clean('updated'),
               'categories': [c.attrib['term'] for c in entry.findall('a:category', NS)],
               'url': 'https://arxiv.org/abs/' + aid, 'metadata_source': 'arXiv API',
               'raw_source': path.relative_to(DATA).as_posix()}
        assert '2020-01-01' <= row['published'][:10] <= '2026-10-11'
        assert aid not in records, ('Repeated ID during pagination', aid)
        records[aid] = row
    responses.append({'url': url, 'raw_source': path.relative_to(DATA).as_posix(),
                      'sha256': hashlib.sha256(content).hexdigest(), 'entries': len(entries),
                      'total_results': total})
    print(f'{start}: {len(entries)} entries; unique {len(records)} / {total}', flush=True)
    if not entries or start + len(entries) >= total:
        break
    start += len(entries)

(DATA / 'discovery' / 'expanded_candidates.jsonl').write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in records.values()), encoding='utf-8')
(DATA / 'discovery' / 'api_search_manifest.json').write_text(
    json.dumps({'query': QUERY, 'collected_at_utc': (datetime.now(timezone.utc).isoformat() if used_network
                                     else expected_search['collected_at_utc']),
                'unique_candidates': len(records), 'responses': responses}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Discovery saved', flush=True)
if not used_network:
    replayed_search = json.loads((DATA / 'discovery' / 'api_search_manifest.json').read_text(encoding='utf-8'))
    assert replayed_search == expected_search
print('Поиск восстановлен из сохранённых XML' if not used_network else 'Получены новые ответы API')

0: 100 entries; unique 100 / 1238
100: 100 entries; unique 200 / 1238
200: 100 entries; unique 300 / 1238
300: 100 entries; unique 400 / 1238
400: 100 entries; unique 500 / 1238
500: 100 entries; unique 600 / 1238
600: 100 entries; unique 700 / 1238
700: 100 entries; unique 800 / 1238
800: 100 entries; unique 900 / 1238
900: 100 entries; unique 1000 / 1238
1000: 100 entries; unique 1100 / 1238
1100: 100 entries; unique 1200 / 1238
1200: 38 entries; unique 1238 / 1238
Discovery saved
Поиск восстановлен из сохранённых XML


In [3]:
original = json.loads((DATA / 'collection_manifest.json').read_text(encoding='utf-8'))
if not (DATA / 'discovery' / 'original_manifest.json').exists():
    (DATA / 'discovery' / 'original_manifest.json').write_text(json.dumps(original, ensure_ascii=False, indent=2), encoding='utf-8')
original = json.loads((DATA / 'discovery' / 'original_manifest.json').read_text(encoding='utf-8'))
original_rows = [json.loads(line) for line in (DATA / 'papers.jsonl').read_text(encoding='utf-8').splitlines()]
baseline_path = DATA / 'discovery' / 'original_papers.jsonl'
if not baseline_path.exists():
    baseline_path.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in original_rows), encoding='utf-8')
original_rows = [json.loads(line) for line in baseline_path.read_text(encoding='utf-8').splitlines()]
expanded = [json.loads(line) for line in (DATA / 'discovery' / 'expanded_candidates.jsonl').read_text(encoding='utf-8').splitlines()]
all_rows = {row['arxiv_id']: row for row in original_rows}
all_rows.update({row['arxiv_id']: row for row in expanded})

PHRASE = re.compile(r'(?:music(?:al)?|song|melody|piano|midi|accompaniment|harmoni[sz]ation)[\s\-\w,]{0,35}(?:generat\w*|compos\w*|synthesi\w*)|(?:generat\w*|compos\w*|synthesi\w*)\b[^.;]{0,75}\b(?:music(?:al)?|songs?|melod(?:y|ies)|piano|midi|accompaniment)\b|text.to.music|singing (?:voice )?synthesis', re.I)
DOMAIN = re.compile(r'\b(?:music\w*|song\w*|melod\w*|piano\w*|midi|harmon\w*|chord\w*|singing|lyric\w*|drum\w*|rhythm\w*|accompan\w*|score\w*|timbre|instrument\w*|guitar\w*|orchestrat\w*|opera)\b', re.I)
WRONG_OUTPUT = re.compile(r'\b(?:dance|dancer|dancing|motion|choreograph\w*|gesture|animation|avatar)\b|video generation', re.I)
MUSIC_OUTPUT = re.compile(r'music generation|song generation|text.to.music|generating (?:symbolic )?music|music synthesis|generation of (?:symbolic )?music|chord generation|musicgen|dance.to.music|gesture.driven music|multimodal music generation', re.I)
OTHER_TASK = re.compile(r'recommendation|recommender|retrieval|playlist|music.to.(?:text|image|video)|music2video|(?:description|comment|caption).*generation|generating (?:captions|descriptions|comments)|singing.head|singing.face|singing.video|guitar tabs generation using ai and audio analysis', re.I)

include_groups = {
    'Neural audio synthesis or continuation applied to music': '2001.00576 2005.05032 2005.08526 2006.14348 2103.07220 2105.01531 2106.06426 2111.05011 2202.07765 2202.09729 2204.07064 2205.07319 2209.03143 2301.13267 2302.13542 2308.05734 2310.00704 2312.15821 2402.00892 2402.15516 2408.00196 2408.09792 2505.09091',
    'Direct neural music interface, notation or rhythm task': '2006.13469 2008.04391 2008.12073 2006.14221 2104.13056 2107.05009 2205.05448 2206.00485 2206.14723 2307.05324 2408.02711 2508.16332 2509.15625',
    'Music source synthesis/separation with a generative model': '2108.03456 2507.01339 2608.01093 2610.11361',
    'Survey of neural audio/music synthesis': '2108.00443 2308.12792 2201.02490 2308.15422',
}
include = {aid: reason for reason, ids in include_groups.items() for aid in ids.split()}
exclude_groups = {
    'Non-neural mathematical, physical or rule-based composition': '2201.10557 2607.05007 2005.05832 2309.12254 2310.07030 2409.07104 2505.20565 2604.20882 2607.06296 2006.03471 2008.04415 2010.01578 2104.12432 2104.13040 2105.14909 2208.06878 2304.04915 2408.11461 2411.19844 2502.15849 2606.19936 2609.11353 2609.34735 2506.08200',
    'Output is lyrics, tips, playlist text or visual content': '2106.01960 2205.06985 2301.08145 2303.14044 2305.07760 2305.19228 2507.04758 2512.23320 2610.03428 2102.00201 2104.11984 2110.07354 2210.00434 2310.09123 2605.17181 2301.11489 2209.01996 2201.03809 2307.03045 2609.22264',
    'Recognition, transcription or unrelated application without a music generator': '2505.09539 2112.07192 2409.02307 2603.10515 2401.16803 2502.12438 2508.07987 2610.11197 2311.08884 2508.04723 2212.01884 2508.05878 2004.10345',
}
exclude = {aid: reason for reason, ids in exclude_groups.items() for aid in ids.split()}
assert not (set(include) & set(exclude))
assert set(include) <= set(all_rows), set(include) - set(all_rows)
assert set(exclude) <= set(all_rows), set(exclude) - set(all_rows)

review, accepted = [], []
for aid, row in all_rows.items():
    title, abstract = row['title'], row['abstract']
    automatic = bool(PHRASE.search(title + ' . ' + abstract) and DOMAIN.search(title))
    if WRONG_OUTPUT.search(title) and not MUSIC_OUTPUT.search(title):
        automatic = False
    if OTHER_TASK.search(title) and not MUSIC_OUTPUT.search(title):
        automatic = False
    decision = (automatic or aid in include) and aid not in exclude
    reason = exclude.get(aid) or include.get(aid) or ('Music-domain title and generation/synthesis context' if decision else 'No direct music-generation scope under the documented rules')
    review.append({'arxiv_id': aid, 'title': title, 'accepted': int(decision),
                   'decision_type': 'reviewed_exception' if aid in include or aid in exclude else 'text_rule', 'reason': reason})
    if decision:
        accepted.append(row)

accepted.sort(key=lambda row: (row['published'], row['arxiv_id']))
assert len(accepted) >= 600
with (DATA / 'selection_review.csv').open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=list(review[0])); writer.writeheader(); writer.writerows(review)
(DATA / 'papers.jsonl').write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in accepted), encoding='utf-8')
(DATA / 'selection_ids.json').write_text(json.dumps([row['arxiv_id'] for row in accepted]), encoding='utf-8')
search_manifest = json.loads((DATA / 'discovery' / 'api_search_manifest.json').read_text(encoding='utf-8'))
manifest = {'collected_at_utc': search_manifest['collected_at_utc'], 'cutoff': '2026-10-11',
            'official_search_query': search_manifest['query'], 'metadata_sources': {'arXiv API': len(accepted)},
            'accepted_count': len(accepted), 'unique_candidates': len(all_rows),
            'original_selection_count': len(original_rows),
            'retained_original': len(set(row['arxiv_id'] for row in accepted) & set(row['arxiv_id'] for row in original_rows)),
            'new_accepted': len(set(row['arxiv_id'] for row in accepted) - set(row['arxiv_id'] for row in original_rows)),
            'discovery_source': 'Official arXiv API search plus the earlier OpenAlex-discovered selection',
            'sampling': 'All results of a fixed title-oriented API query; documented phrase rules and reviewed inclusion/exclusion exceptions. Not an exhaustive catalogue.',
            'selection_note': 'Heuristic corpus around neural music generation: methods, controls, datasets, evaluation, comparative approaches and adjacent music-processing tasks. Known major off-topic outputs are filtered by documented rules and exceptions; this is not a gold-labeled catalogue of generators.',
            'api_responses': original['api_responses'] + search_manifest['responses']}
(DATA / 'collection_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({key: value for key, value in manifest.items() if key != 'api_responses'}, ensure_ascii=False, indent=2))
assert accepted == expected_papers, 'Отбор отличается от корпуса в основном ноутбуке'
assert (DATA / 'selection_review.csv').read_text(encoding='utf-8-sig') == expected_review
assert manifest == expected_manifest
print(f'Проверка отбора: {len(review)} кандидатов → {len(accepted)} публикации')

{
  "collected_at_utc": "2026-10-11T14:10:02.737994+00:00",
  "cutoff": "2026-10-11",
  "official_search_query": "(ti:music OR abs:music OR ti:musical OR abs:musical) AND (ti:generation OR ti:generative OR ti:generating OR ti:composition OR ti:composing OR ti:synthesis OR ti:composer) AND submittedDate:[202001010000 TO 202610112359]",
  "metadata_sources": {
    "arXiv API": 793
  },
  "accepted_count": 793,
  "unique_candidates": 1283,
  "original_selection_count": 350,
  "retained_original": 333,
  "new_accepted": 460,
  "discovery_source": "Official arXiv API search plus the earlier OpenAlex-discovered selection",
  "sampling": "All results of a fixed title-oriented API query; documented phrase rules and reviewed inclusion/exclusion exceptions. Not an exhaustive catalogue.",
  "selection_note": "Heuristic corpus around neural music generation: methods, controls, datasets, evaluation, comparative approaches and adjacent music-processing tasks. Known major off-topic outputs are filter

In [ ]:
NS = {'a': 'http://www.w3.org/2005/Atom'}
manifest = json.loads((DATA / 'collection_manifest.json').read_text(encoding='utf-8'))
selection = json.loads((DATA / 'selection_ids.json').read_text(encoding='utf-8'))
with (DATA / 'selection_review.csv').open(encoding='utf-8-sig', newline='') as file:
    decisions = list(csv.DictReader(file))
assert set(selection) == {row['arxiv_id'] for row in decisions if row['accepted'] == '1'}
records = {}
for item in manifest['api_responses']:
    path = DATA / item['raw_source']
    if not path.exists():
        request = urllib.request.Request(item['url'], headers={'User-Agent': 'ITMO-MusicGenerationLab/1.0',
                                                               'Accept': 'application/atom+xml'})
        with urllib.request.urlopen(request, timeout=45) as response:
            content = response.read()
        ET.fromstring(content)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
        time.sleep(3.5)
    content = path.read_bytes()
    # Новая версия из API может отличаться от сохранённого снимка.
    assert hashlib.sha256(content).hexdigest() == item['sha256'], 'API response differs from the saved snapshot'
    entries = ET.fromstring(content).findall('a:entry', NS)
    assert len(entries) == item['entries']
    for entry in entries:
        match = re.search(r'(\d{4}\.\d{4,5})(v\d+)?$', entry.findtext('a:id', '', NS))
        aid = match.group(1)
        if aid not in selection:
            continue
        text = lambda tag: ' '.join(entry.findtext('a:' + tag, '', NS).split())
        records[aid] = {'arxiv_id': aid, 'arxiv_version': match.group(2), 'title': text('title'),
                        'authors': [author.findtext('a:name', '', NS) for author in entry.findall('a:author', NS)],
                        'abstract': text('summary'), 'published': text('published'), 'updated': text('updated'),
                        'categories': [category.attrib['term'] for category in entry.findall('a:category', NS)],
                        'url': 'https://arxiv.org/abs/' + aid, 'metadata_source': 'arXiv API',
                        'raw_source': item['raw_source']}
assert set(records) == set(selection) and len(records) == manifest['accepted_count']
assert all('2020-01-01' <= row['published'][:10] <= manifest['cutoff'] for row in records.values())
rows = [records[aid] for aid in selection]
(DATA / 'papers.jsonl').write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')
print(f'Restored {len(rows)} official API records from {len(manifest["api_responses"])} feeds.')
assert rows == expected_papers

Restored 793 official API records from 17 feeds.
Корпус восстановлен без изменения метаданных и порядка записей
